In [ ]:
import pandas as pd
pd.set_option('display.max_rows', 10000)

## Summary

The purpose of this notebook, is to perform the data transformations to the ALA Occurrences Dataset.

Please run each block.

Ensure that below are downloaded and placed in the same directory. 

* AggregatedData_AustralianSpeciesOccurrences_1.1.2023-06-13.csv - https://data.sciro.au/collection/csiro:64131
* Ibra7 Shape Files https://data.gov.au/data/dataset/9791362e-bfb3-4d13-7a7a-dd10f25c4d84
* EPBC Conservation Status: 27082025-093637-report.csv https://www.environment.gov.au/sprat-public/action/report;jsessionid=615A8A17ECE46F24E0BEB892532D9E14 

## 1. Occurrences Data - used for map


In [ ]:
## download from https://data.sciro.au/collection/csiro:64131
occur_df = pd.read_csv("AggregatedData_AustralianSpeciesOccurrences_1.1.2023-06-13.csv")

In [4]:
occur_df.shape

(11619897, 13)

In [342]:
occur_df.isna().sum()

year                       0
basisOfRecord              0
stateTerritory       1089289
ibraRegion           1088674
imcraRegion         10509712
forest2018Status           0
forest2013Status           0
capadStatus                0
epbcStatus                 0
griisStatus                0
speciesID                  0
speciesName                0
occurrenceCount            0
dtype: int64

In [78]:
occur_df.groupby('year').size().reset_index(drop = False)

,year,0
0,1900,15468
1,1901,9334
2,1902,8741
3,1903,10888
4,1904,9350
5,1905,9053
6,1906,9741
7,1907,9243
8,1908,9202
9,1909,11331


In [ ]:
occur_df_2022 = occur_df.query("year == 2022")

In [29]:
occur_df_2022.head()

,year,basisOfRecord,stateTerritory,ibraRegion,imcraRegion,forest2018Status,forest2013Status,capadStatus,epbcStatus,griisStatus,speciesID,speciesName,occurrenceCount
11317553,2022,HUMAN_OBSERVATION,Ashmore and Cartier Islands,Indian Tropical Islands,NaN,non-forest,non-forest,PA,Not listed,Native,NZOR-6-24112,Gallirallus philippensis,1
11317554,2022,HUMAN_OBSERVATION,Ashmore and Cartier Islands,Indian Tropical Islands,NaN,non-forest,non-forest,PA,Not listed,Native,https://biodiversity.org.au/afd/taxa/00cf9ec9-...,Anous stolidus,2
11317555,2022,HUMAN_OBSERVATION,Ashmore and Cartier Islands,Indian Tropical Islands,NaN,non-forest,non-forest,PA,Not listed,Native,https://biodiversity.org.au/afd/taxa/13a31834-...,Onychoprion fuscatus,1
11317556,2022,HUMAN_OBSERVATION,Ashmore and Cartier Islands,Indian Tropical Islands,NaN,non-forest,non-forest,PA,Not listed,Native,https://biodiversity.org.au/afd/taxa/1441c509-...,Thalasseus bergii,1
11317557,2022,HUMAN_OBSERVATION,Ashmore and Cartier Islands,Indian Tropical Islands,NaN,non-forest,non-forest,PA,Not listed,Native,https://biodiversity.org.au/afd/taxa/31a2070b-...,Sula leucogaster,1


In [63]:
occur_df_2022.groupby(['speciesName', 'ibraRegion']).agg({"occurrenceCount": "sum"}).reset_index().sort_values("occurrenceCount")

,speciesName,ibraRegion,occurrenceCount
97664,Solanum chippendalei,Tanami,1
46912,Gazania krebsiana,South East Coastal Plain,1
46913,Gazania linearis,Cobar Peneplain,1
46914,Gazania linearis,Eyre Yorke Block,1
46916,Gazania linearis,Geraldton Sandplains,1
...,...,...,...
76165,Osphranter rufus,Murray Darling Depression,1994
81287,Phascolarctos cinereus,South Eastern Queensland,2007
5155,Alternanthera denticulata,Riverina,2069
91193,Rattus fuscipes,Sydney Basin,2272


## 2. Conservation Status

In [46]:
cons_status = pd.read_csv("27082025-093637-report.csv")

In [47]:
cons_status.head()

,TaxonID,ScientificName,CommonName,EPBCThreatStatus,EPBCThreatenedSpeciesListedName,EPBCThreatenedSpeciesDateEffective,ConservationAdvice,Kingdom,Family,AustralianCapitalTerritory,...,WesternAustralia,AshmoreandCartierIslands,Cocos(Keeling)Islands,ChristmasIsland,CoralSeaIslands,JervisBayTerritory,NorfolkIsland,HeardandMcDonaldIslands,AustralianAntarcticTerritory,CommonwealthMarineArea
0,66785,Abebaioscia troglodytes,Pannikin Plains Cave Isopod,NaN,NaN,NaN,NaN,Animalia,Philosciidae,NaN,...,Present,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,93331,Abrodictyum obscurum,,NaN,NaN,NaN,NaN,Plantae,Hymenophyllaceae,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,8929,Abutilon fraseri,Dwarf Lantern-flower,NaN,NaN,NaN,NaN,Plantae,Malvaceae,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,27797,Abutilon julianae,Norfolk Island Abutilon,Critically Endangered,Abutilon julianae,25-Nov-03,NaN,Plantae,Malvaceae,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Endemic,NaN,NaN,NaN
4,2204,Abutilon malvifolium,"Mallow-leaf Lantern-flower, Bastard Marshmallow",NaN,NaN,NaN,NaN,Plantae,Malvaceae,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [99]:
cons_status['EPBCThreatStatus'] = cons_status['EPBCThreatStatus'].fillna("Present")

In [133]:
df = occur_df.merge(cons_status[['ScientificName','EPBCThreatStatus', "CommonName", "Kingdom"]], left_on = "speciesName", right_on = "ScientificName")

In [134]:
df[df["EPBCThreatStatus"].notnull()].EPBCThreatStatus.value_counts()

Present                   1180658
Vulnerable                 128753
Endangered                  64304
Critically Endangered       26162
Conservation Dependent       1163
Extinct                       433
Extinct in the wild            21
Name: EPBCThreatStatus, dtype: int64

In [135]:
df.head()

,year,basisOfRecord,stateTerritory,ibraRegion,imcraRegion,forest2018Status,forest2013Status,capadStatus,epbcStatus,griisStatus,speciesID,speciesName,occurrenceCount,ScientificName,EPBCThreatStatus,CommonName,Kingdom
0,1900,HUMAN_OBSERVATION,New South Wales,Australian Alps,NaN,forest,forest,PA,Critically Endangered,Native,https://biodiversity.org.au/afd/taxa/1a3bc394-...,Pseudophryne corroboree,3,Pseudophryne corroboree,Critically Endangered,Southern Corroboree Frog,Animalia
1,1900,HUMAN_OBSERVATION,New South Wales,Australian Alps,NaN,forest,non-forest,PA,Critically Endangered,Native,https://biodiversity.org.au/afd/taxa/1a3bc394-...,Pseudophryne corroboree,1,Pseudophryne corroboree,Critically Endangered,Southern Corroboree Frog,Animalia
2,1900,HUMAN_OBSERVATION,New South Wales,Australian Alps,NaN,non-forest,forest,PA,Critically Endangered,Native,https://biodiversity.org.au/afd/taxa/1a3bc394-...,Pseudophryne corroboree,1,Pseudophryne corroboree,Critically Endangered,Southern Corroboree Frog,Animalia
3,1900,HUMAN_OBSERVATION,New South Wales,Australian Alps,NaN,non-forest,non-forest,PA,Critically Endangered,Native,https://biodiversity.org.au/afd/taxa/1a3bc394-...,Pseudophryne corroboree,1,Pseudophryne corroboree,Critically Endangered,Southern Corroboree Frog,Animalia
4,1950,HUMAN_OBSERVATION,New South Wales,Australian Alps,NaN,forest,non-forest,PA,Critically Endangered,Native,https://biodiversity.org.au/afd/taxa/1a3bc394-...,Pseudophryne corroboree,1,Pseudophryne corroboree,Critically Endangered,Southern Corroboree Frog,Animalia


In [ ]:
df2 = (df[df["EPBCThreatStatus"]
          .notnull()]
       .groupby(["year","stateTerritory", "ibraRegion", "ScientificName", "CommonName", "Kingdom", "griisStatus", "capadStatus", "EPBCThreatStatus"])
       ["occurrenceCount"]
       .sum()
       .reset_index()
       .query("Kingdom == 'Animalia'"))

In [137]:
df2.head()

,year,stateTerritory,ibraRegion,ScientificName,CommonName,Kingdom,griisStatus,capadStatus,EPBCThreatStatus,occurrenceCount
0,1900,New South Wales,Australian Alps,Ornithorhynchus anatinus,Platypus,Animalia,Native,PA,Present,1
2,1900,New South Wales,Australian Alps,Pseudophryne corroboree,Southern Corroboree Frog,Animalia,Native,PA,Critically Endangered,6
5,1900,New South Wales,Brigalow Belt South,Ardeotis australis,Australian Bustard,Animalia,Native,not protected,Present,2
6,1900,New South Wales,Brigalow Belt South,Chlamydera maculata,Spotted Bowerbird,Animalia,Native,not protected,Present,1
8,1900,New South Wales,Brigalow Belt South,Hemiaspis damelii,Grey Snake,Animalia,Native,not protected,Endangered,3


In [184]:
## save to csv
df2.to_csv('wildlife_occurrences_with_location_end_status_v1.csv', index=False)

In [177]:
agg_2023 = df2.query("year == 2022 and griisStatus == 'Native'").groupby(["year","CommonName", "ScientificName", "EPBCThreatStatus"])["occurrenceCount"].sum().reset_index().sort_values(["year", "occurrenceCount"], ascending = False)
# get list of present and endangered animals 
print(list(agg_2023.query("EPBCThreatStatus == 'Present'")["CommonName"].head(30)))
print(list(agg_2023.query("EPBCThreatStatus == 'Endangered'")["CommonName"].head(20)))

['Rabbit, European Rabbit', 'Red Fox, Fox', 'Koala', 'Eastern Grey Kangaroo', 'Swamp Wallaby', 'Short-beaked Echidna', 'Western Grey Kangaroo', 'Red Kangaroo', 'Bare-nosed Wombat, Common Wombat', 'Silver Gull', 'Yellow-tailed Black-Cockatoo', 'Platypus', 'Southern Long-nosed Bandicoot', 'Brown-striped Frog, Striped Marsh Frog', 'Eastern Long-necked Turtle', 'Cane Toad', 'Agile Antechinus', 'Diamond Python', 'Bush Stone-curlew', 'Murray Turtle, Macquarie Tortoise', 'Scarlet Robin', 'Great Egret, White Egret', 'Yellow-footed Antechinus', 'Rainbow Bee-eater', 'White-bellied Sea-Eagle', 'Glossy Black-Cockatoo', 'Burrowing Bettong', 'Yellow-bellied Glider', 'Brown Treecreeper', 'Flame Robin']
['Gang-gang Cockatoo', 'Greater Glider (southern and central)', 'Blue Mountains Water Skink', 'Southern Cassowary', 'Large-eared Pied Bat, Large Pied Bat', 'Long-footed Potoroo', "Baudin's Cockatoo, Baudin's Black-Cockatoo, Long-billed Black-cockatoo", 'Bridled Nail-tail Wallaby, Bridled Nailtail Walla

## 3. Adding Latitude/Longitude from Ibra7


In [213]:
import fiona
from shapely.geometry import shape
from shapely.ops import transform
from pyproj import Transformer
import csv

# Input shapefile
shapefile = "IBRARegion_Aust70.shp"

# Output CSV
output_csv = "ibra7_centroids_latlon.csv"

with fiona.open(shapefile) as src:
    # Build transformer from source CRS to WGS84 (EPSG:4326)
    transformer = Transformer.from_crs(src.crs, "EPSG:4326", always_xy=True)

    with open(output_csv, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["IBRA_REG_N", "STATE", "lat", "lon"])
        
        for feature in src:
            geom = shape(feature["geometry"])
            centroid = geom.centroid

            # Reproject centroid to lat/lon
            lon, lat = transformer.transform(centroid.x, centroid.y)

            reg_name = feature["properties"]["IBRA_REG_N"]
            state = feature["properties"]["STATE"]

            writer.writerow([reg_name, state, lat, lon])

print(f"✅ Saved centroids in proper lat/lon to {output_csv}")

✅ Saved centroids in proper lat/lon to ibra7_centroids_latlon.csv


In [296]:
df2.head()

,year,stateTerritory,ibraRegion,ScientificName,CommonName,Kingdom,griisStatus,capadStatus,EPBCThreatStatus,occurrenceCount
0,1900,NSW,Australian Alps,Ornithorhynchus anatinus,Platypus,Animalia,Native,PA,Present,1
2,1900,NSW,Australian Alps,Pseudophryne corroboree,Southern Corroboree Frog,Animalia,Native,PA,Critically Endangered,6
5,1900,NSW,Brigalow Belt South,Ardeotis australis,Australian Bustard,Animalia,Native,not protected,Present,2
6,1900,NSW,Brigalow Belt South,Chlamydera maculata,Spotted Bowerbird,Animalia,Native,not protected,Present,1
8,1900,NSW,Brigalow Belt South,Hemiaspis damelii,Grey Snake,Animalia,Native,not protected,Endangered,3


In [291]:
state_map = {
    "New South Wales": "NSW",
    "Victoria": "VIC",
    "Queensland": "QLD",
    "South Australia": "SA",
    "Western Australia": "WA",
    "Tasmania": "TAS",
    "Australian Capital Territory": "ACT",
    "Northern Territory": "NT",
    "Jervis Bay Territory": "JBT",
    "External Territories": "EXT"
}

# Replace full names with abbreviations
df2["stateTerritory"] = df2["stateTerritory"].replace(state_map)

In [216]:
ipra7_lat_long =  pd.read_csv("ibra7_centroids_latlon.csv")

In [217]:
ipra7_lat_long.head()

,IBRA_REG_N,STATE,lat,lon
0,Australian Alps,ACT,-35.667875,148.909459
1,South Eastern Highlands,ACT,-35.434530,149.030591
2,Coral Sea,EXT,-18.202475,151.620450
3,Indian Tropical Islands,EXT,-11.742666,112.798183
4,Pacific Subtropical Islands,EXT,-29.931893,165.098390


In [292]:
df4 = df2.merge(
    ipra7_lat_long,
    left_on=["ibraRegion", "stateTerritory"],
    right_on=["IBRA_REG_N", "STATE"],
    how="left"   # left join so you don’t drop df2 rows if no match
)

# df4["CommonName_clean"] = (
#     df4["CommonName"]
#       .fillna("")
#       .astype(str)
#       .str.split(",")
#       .str[0]
#       .str.strip()
# )

In [234]:
## save to csv
df4.to_csv('wildlife_occurrences_with_location_end_status_v2_w_lat_long.csv', index=False)

In [343]:
df4 = pd.read_csv('wildlife_occurrences_with_location_end_status_v2_w_lat_long.csv')
df4.head()

,year,stateTerritory,ibraRegion,ScientificName,CommonName,Kingdom,griisStatus,capadStatus,EPBCThreatStatus,occurrenceCount,IBRA_REG_N,STATE,lat,lon,CommonName_clean
0,1900,NSW,Australian Alps,Ornithorhynchus anatinus,Platypus,Animalia,Native,PA,Present,1,Australian Alps,NSW,-36.074960,148.469095,Platypus
1,1900,NSW,Australian Alps,Pseudophryne corroboree,Southern Corroboree Frog,Animalia,Native,PA,Critically Endangered,6,Australian Alps,NSW,-36.074960,148.469095,Southern Corroboree Frog
2,1900,NSW,Brigalow Belt South,Ardeotis australis,Australian Bustard,Animalia,Native,not protected,Present,2,Brigalow Belt South,NSW,-30.762574,149.581615,Australian Bustard
3,1900,NSW,Brigalow Belt South,Chlamydera maculata,Spotted Bowerbird,Animalia,Native,not protected,Present,1,Brigalow Belt South,NSW,-30.762574,149.581615,Spotted Bowerbird
4,1900,NSW,Brigalow Belt South,Hemiaspis damelii,Grey Snake,Animalia,Native,not protected,Endangered,3,Brigalow Belt South,NSW,-30.762574,149.581615,Grey Snake
